In [9]:
import os
import sys
import time
import yaml
import pandas as pd
import json
from string import Template

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import llm_tools as llt
from utils import is_casenum, extract_json

In [10]:
MODEL = "claude-opus-4-6"
LLM_PROVIDER = "claude" # openai | claude
MAX_OUTPUT_TOKENS = 4000
input_cost = 2.50 # batch cost per 1M tokens
output_cost = 12.50 # batch cost per 1M tokens
output_tokens_per_request = 500
REQUESTS_PER_BATCH = 200

In [11]:
MIN_IDX = 0
MAX_IDX = 250
REPLACE = False
REPLACE_OUTPUT_FILE = True
OUTPUT_FILENAME = "supplemental-docs-summarized-2.json"
MODE = 'WRITE'  # ESTIMATE | BATCH | WRITE

In [12]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [13]:
PROMPT = Template("""
You are an analyst working for the Los Angeles City Planning Commission. Your task is to read the following meeting agenda item and a letter or petition submitted by a member of the public which $support_oppose the proposed project. You will then identify the reasons for the support or opposition.

Return valid JSON only. Do not include any other text, markdown formatting, or code fences.

## Output Format

Here is an example of the expected output:

{
    "reasons": [list of categorical reasons for support or opposition; see list of possible reasons below],
    "explanation": a brief explanation for why you chose the listed reasons
}


## List of possible reasons

- "HOUSING AFFORDABILITY / AFFORDABLE HOUSING"
- "HOUSING SUPPLY"
- "TRAFFIC IMPACT"
- "NEIGHBORHOOD CHARACTER"
- "TRANSIT ACCESS"
- "DISPLACEMENT / GENTRIFICATION"
- "WALKABILITY / BIKEABILITY"
- "HOMELESSNESS"
- "CLIMATE / ENVIRONMENT"
- "PROCEDURAL MATTERS"
- "ECONOMIC IMPACTS / JOBS"
- "FAIR HOUSING / HOUSING EQUITY"
- "PARKING IMPACT"
- "HISTORIC / CULTURAL PRESERVATION"
- "SAFETY"
- "NOISE / LIGHT POLLUTION"
- "CEQA RELATED MATTERS"
- "BUILDING HEIGHT / SIZE / DENSITY"
- "COMMUNITY ENGAGEMENT / OPPOSITION"
- "UNION LABOR / PREVAILING WAGES"

## Guidelines

- Choose only from the listed reasons
- "TOC" stands for Transit-Oriented-Communities and is related to TRANSIT ACCESS.
- "MND" stands for Mitigated Negative Declaration, "CE" stands for Categorical Exemption, "ND" stands for Negative Declaration; all are related to CEQA.


<agenda_item>

$agenda_text

</agenda_item>

<supplemental_document>

$document_text

</supplemental_document>
""")

In [14]:
# write prompt to figures
with open(os.path.join(LOCAL_PATH, 'figures', 'support_oppose_reasons_prompt.tex'), 'w') as f:
    out = PROMPT.substitute(support_oppose="...", agenda_text="...", document_text="...")
    out = out.strip()
    out = out.replace('\n', '\\\\ \n')
    f.write(out)

In [15]:
input_tokens = 0
output_tokens = 0
n_requests = 0
n_direct = 0
batch_num = 0
max_input_token_length = 0
max_input_token_date = ""
max_input_token_prompt = ""

rel_path = os.path.join(DATA_PATH, "response_store")
batch_db_path = os.path.join(rel_path, "batch_jobs.db")
response_db_path = os.path.join(rel_path, "responses.db")

batch_db_conn = llt.get_batch_jobs_db_conn(batch_db_path)
response_db_conn = llt.get_response_store_db_conn(response_db_path)

errored = []

t0 = time.time()
prompts_to_submit = []
for idx, row in meetings_df.iterrows():
    if idx < MIN_IDX or idx > MAX_IDX:
        continue

    date = row['date']
    year = date[0:4]

    print(f"{idx} {date} ...")

    # Agenda json
    agenda_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized-2.json")
    if not os.path.exists(agenda_filepath):
        print("No agenda text file found, skipping.")
        continue
    with open(agenda_filepath, 'r', encoding='utf-8') as f:
        agenda_json = json.load(f)

    # Supplemental document df
    docs_df_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "supplemental-docs.pkl")
    if not os.path.exists(docs_df_filepath):
        print("No supplemental documents file found, skipping.")
        continue
    docs_df = pd.read_pickle(docs_df_filepath)

    # Supplemental docs json
    docs_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "supplemental-docs-summarized.json")
    if not os.path.exists(docs_filepath):
        print("No supplemental docs file found, skipping.")
        continue
    with open(docs_filepath, 'r', encoding='utf-8') as f:
        docs_json = json.load(f)

    # Output paths
    output_dir = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date)
    os.makedirs(output_dir, exist_ok=True)
    output_filepath = os.path.join(output_dir, OUTPUT_FILENAME)
    if (not REPLACE_OUTPUT_FILE) and os.path.exists(output_filepath):
        print("Output file exists, skipping.")
        continue

    # Iterate
    docs_json_2 = docs_json.copy()
    for _i_, doc in enumerate(docs_json):
        doc_id = doc['doc_id']
        document_text = docs_df.loc[docs_df["doc_id"]==doc_id, "content"].values[0]
        docs_json_2[_i_]["support_or_oppose_reasons"] = {}
        docs_json_2[_i_]["support_or_oppose_explanation"] = {}
        
        for item_no, support_or_oppose in doc["support_or_oppose"].items():
            if support_or_oppose in ["DEFINITELY SUPPORT", "SOMEWHAT SUPPORT"]:
                support_or_oppose_text = "supports"
            elif support_or_oppose in ["DEFINITELY OPPOSE", "SOMEWHAT OPPOSE"]:
                support_or_oppose_text = "opposes"
            else:
                continue # skip if neutral or N/A

            # loop for agenda_json to find matching agenda item
            agenda_text = ""
            for agenda_item in agenda_json:
                if agenda_item["item_no"] == item_no:
                    agenda_text = agenda_item["text"]
                    break
            if agenda_text == "":
                continue
            
            prompt = PROMPT.substitute(
                support_oppose = support_or_oppose_text,
                agenda_text = agenda_text,
                document_text = document_text   
            )
            
            # check if prompt is in cache
            if not REPLACE:
                prompt_hash = llt.get_hash(prompt)
                cached_response = llt.check_response_cache(prompt_hash, response_db_conn)
                if cached_response:
                    if MODE=='WRITE':
                        response = cached_response[0]
                        logprob = cached_response[1]
                        perplexity = cached_response[2]
                        j = extract_json(response)
                        if j is None:
                            errored.append((date, doc_id))
                            print(f"    cached response for {date} doc_id {doc_id} is not valid JSON, adding to errored list")
                        else:
                            docs_json_2[_i_]["support_or_oppose_reasons"][item_no] = j["reasons"]
                            docs_json_2[_i_]["support_or_oppose_explanation"][item_no] = j["explanation"]
                    continue

            #my_tokens = llt.count_tokens(prompt, MODEL)
            input_tokens += len(prompt)/3.5
            output_tokens += output_tokens_per_request
            if len(prompt)/3.5 > max_input_token_length:
                max_input_token_length = len(prompt)/3.5
                max_input_token_date = date
                max_input_token_prompt = prompt
            n_requests += 1

            # add prompt to batch and check if batch is ready to submit
            if MODE=='BATCH':
                prompts_to_submit.append(prompt)
                if len(prompts_to_submit) >= REQUESTS_PER_BATCH:
                    input_filename = f"support_oppose_reasons_batch_{batch_num}.jsonl"
                    llt.create_chat_batch_job(prompts_to_submit, rel_path, input_filename, model=MODEL, batch_db_conn=batch_db_conn, response_db_conn=response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)
                    prompts_to_submit = []
                    batch_num += 1
                    print(f"    batch submitted")
            elif MODE=='WRITE':
                try:
                    my_response = llt.get_response(prompt, MODEL, response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)
                except:
                    print(f"Error getting response for {date} doc_id {doc_id}, adding to errored list")
                    errored.append((date, doc_id))
                    continue
                response = my_response['response']
                logprob = my_response['total_logprob']
                perplexity  = my_response['perplexity']
                n_direct += 1
                j = extract_json(response)
                if j is None:
                    errored.append((date, doc_id))
                    print(f"    response for {date} doc_id {doc_id} is not valid JSON, adding to errored list")
                else:
                    docs_json_2[_i_]["support_or_oppose_reasons"][item_no] = j["reasons"]
                    docs_json_2[_i_]["support_or_oppose_explanation"][item_no] = j["explanation"]
    
    if MODE=='WRITE':
        # write output file
        with open(output_filepath, 'w') as f:
            json.dump(docs_json_2, f, indent=2)
        print(f"    wrote output file")        

if (MODE=='BATCH') and (len(prompts_to_submit) > 0):
    input_filename = f"support_oppose_reasons_batch_{batch_num}.jsonl"
    llt.create_chat_batch_job(prompts_to_submit, rel_path, input_filename, model=MODEL, batch_db_conn=batch_db_conn, response_db_conn=response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)
    prompts_to_submit = []
    print(f"    batch submitted")

t1 = time.time()
print(f"    elapsed time: {(t1-t0)/60:.2f} minutes, {n_direct:,} direct requests")

batch_db_conn.close()
response_db_conn.close()

print(f"{n_direct:,} direct requests made")

0 2018-05-10 ...
    wrote output file
1 2018-05-23 ...
    wrote output file
2 2018-06-14 ...
    wrote output file
3 2018-07-12 ...
    wrote output file
4 2018-07-26 ...
    wrote output file
5 2018-08-09 ...
    wrote output file
6 2018-08-23 ...
    wrote output file
7 2018-09-13 ...
    wrote output file
8 2018-09-27 ...
    wrote output file
9 2018-10-11 ...
    wrote output file
10 2018-10-25 ...
    wrote output file
11 2018-11-08 ...
    wrote output file
12 2018-11-29 ...
    wrote output file
13 2018-12-13 ...
    wrote output file
14 2018-12-20 ...
    wrote output file
15 2019-01-10 ...
    wrote output file
16 2019-01-24 ...
    wrote output file
17 2019-02-14 ...
    wrote output file
18 2019-02-28 ...
    wrote output file
19 2019-03-14 ...
    wrote output file
20 2019-03-28 ...
    wrote output file
21 2019-04-11 ...
    wrote output file
22 2019-05-09 ...
    wrote output file
23 2019-05-23 ...
    wrote output file
24 2019-06-13 ...
    wrote output file
25 2019-06

In [16]:
# Cost Estimate

total_input_cost = input_tokens / 1e6 * input_cost
total_output_cost = output_tokens / 1e6 * output_cost

print(f"Estimated number of requests: {n_requests:,.0f}")
print(f"Estimated total input tokens: {input_tokens:,.0f}")
print(f"Estimated total input cost: ${total_input_cost:.2f}")
print(f"Estimated total output tokens: {n_requests * output_tokens_per_request:,.0f}")
print(f"Estimated total output cost: ${total_output_cost:.2f}")
print(f"Estimated total cost: ${total_input_cost + total_output_cost:.2f}")
print(f"Max input token length: {max_input_token_length:,.0f} on date {max_input_token_date}")

Estimated number of requests: 10
Estimated total input tokens: 139,554
Estimated total input cost: $0.35
Estimated total output tokens: 5,000
Estimated total output cost: $0.06
Estimated total cost: $0.41
Max input token length: 25,103 on date 2022-03-10
